# Jupyter Notebook: Decision Trees

## Step 1: Install Required Libraries

First, ensure you have the necessary libraries installed. You can install them using the following command:

`pip install pandas numpy seaborn matplotlib scikit-learn`

In [ ]:
import matplotlib.pyplot as plt
# Imports
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

## Step 2: Reading the data and data cleaning
Reading the training data

In [ ]:
df = pd.read_csv(r'train.csv')  # Replace by your path
df.head(5)

Determining how many missing values each column contains. 

In [ ]:
column_names = df.columns
for column in column_names:
    missing_count = df[column].isnull().sum()
    total_count = len(df[column])
    missing_percentage = (missing_count / total_count) * 100
    print(f'{column} - {missing_count}, {missing_percentage:.2f}%')

* Age has ~20% missing values. This will require handling.
* Cabin has ~77% missing values and might be excluded from analysis.
* Embarked has a small proportion of missing values. This will require handling.

Filling missing values for 'age' with the median. The median is used here instead of the mean because it is less affected by outliers. The 'age' column might have extreme values (very young or very old passengers), and the median provides a more robust measure.

In [ ]:
df['Age'] = df['Age'].fillna(df['Age'].median())

Filling missing values for 'Embarked' with the mode.
The 'Embarked' column contains categorical data. Using the mode (most frequent value) ensures that the imputed value makes sense contextually.

In [ ]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

Dropping 'Cabin' due to a large number of missing values
The 'Cabin' column has too many missing values to be useful for analysis. Dropping it simplifies the dataset.

In [ ]:
df = df.drop(columns=['Cabin'])

Let's make sure that there are no more missing values. 

In [ ]:
column_names = df.columns
for column in column_names:
    missing_count = df[column].isnull().sum()
    total_count = len(df[column])
    missing_percentage = (missing_count / total_count) * 100
    print(f'{column} - {missing_count}, {missing_percentage:.2f}%')

In [ ]:
# Dropping the passengerId, Name, and Ticket
df = df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)

In [ ]:
# Convert categorical to numeric
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

## Step 3: Separating the target value (Survived) from the rest 
Separating the features from the target

In [ ]:
# Features and target
X = df.drop('Survived', axis=1)
y = df['Survived']

## Step 4: Spliting the data 
Dividing the data into train and validation sets (Kaggle has already separated a test set) 

In [ ]:
# Train-val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 5: Training the decision tree classifier
Defining the model and training the model. 

In [ ]:
# Create and train the model
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# Step 6: Calculate the performance of the model
We are going to use some numerical metrics to calculate the performance of the algorithm

In [ ]:

# Predictions
y_pred = model.predict(X_val)

# Evaluation
print("Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_pred))

# Step 7: Visualize the decision tree classifier

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))
plot_tree(model, filled=True, feature_names=X.columns, class_names=['Not Survived', 'Survived'])
plt.title("Decision Tree")
plt.show()

## Investigation tasks

**Task 1**: Explain how you can interpret a confusion matrix and interpret the confusion matrix obtained in step 6.

<span style="color:red">(1 mark)</span>

               precision    recall  f1-score   support

           0       0.82      0.80      0.81       105
           1       0.73      0.76      0.74        74

    accuracy                           0.78       179
   macro avg       0.78      0.78      0.78       179
weighted avg       0.78      0.78      0.78       179


Confusion Matrix:
 [[84 21]
 [18 56]]

The confusion matrix is can be interpreted via the first column/row (1,1) being the amount of positives being predicted as positive, the (1,2) spot means the amount of Negative being positive, (2,1) is the amount of negative being positive, (2,2) is the amount of negative being predicted negative.

**Task 2**: In machine learning, various metrics are used to evaluate the performance of classification algorithms. Two of the most commonly used are Accuracy and F1-Score.
Research and explain what each of these metrics represents, including their mathematical formulas. Then, discuss which metric is more appropriate when dealing with an imbalanced dataset (e.g., when the number of survivors and non-survivors is not equal, as in the Titanic dataset). Provide a clear justification for your choice.

<span style="color:red">(2 marks)</span>

Accuracy is the proportion of correct prediction out of all predictions, it is below:
$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$
F1 Score is a metric that combines precision and recall score, scored between 0 and 1, being calculated from the below formula:
$$
F_1 = \frac{2 \cdot (\text{precision} \cdot \text{recall})}{\text{precision} + \text{recall}}
$$
In this case, the most correct data to pick is F1 score, as a model with high accuracy could be returning undesired predictions in a imbalanced data set, like always returning False.

**Task 3**: Calculate the Accuracy and F1-Score for both the training and validation datasets. Then, analyse and interpret the results, what do the metrics suggest about your model’s performance? Is there any evidence of overfitting or underfitting?

<span style="color:red">(2 marks)</span>

The model accuracy is 0.78, with it being good but not great, also alluding to it performing better on the majority class than the minority class. The f1 score is 0.81 on those whom did not survive, which is pretty good however it is 0.74 on those whom survived, therefore impling that there is evidence of overfitting.

**Task 4**: Train a new decision tree using only 'Sex_male', and 'Age' as features. Calculate and interpret the performance on training and validation. Compare the results to the previous model.

<span style="color:red">(2 marks)</span>

In [ ]:
X = df[['Sex_male', 'Age']]
y = df['Survived']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)


**Task 5**: Visualize new decision tree and interpret it. 

<span style="color:red">(1 mark)</span>

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))
plot_tree(model, filled=True, max_depth=2, feature_names=X.columns, class_names=['Not Survived', 'Survived'])
plt.title("Decision Tree")
plt.show()

The decision tree works via classifying and splitting the data set up, the first question is the most important question, as seen above the question was in regards to its gender, asking whether it is male or female.

**Task 6**: When training a Decision Tree Classifier, several parameters can be adjusted to control the model’s complexity and improve its performance, particularly in addressing issues like overfitting and underfitting.

Using the [scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html), choose three parameters of the DecisionTreeClassifier. For each parameter:

- Describe what it does

- Explain how changing its value could influence overfitting or underfitting in the model

<span style="color:red">(3 marks)</span>

1. Max Depth: What it does is that it limits the maximum depth of the tree. Changing this parameter will affect how big the tree can grow, if it is too big it can lead to overfitting, if it is too small it can lead to underfitting.
2. Min Samples Split: It prevents splitting on small sample sizes, controlling the minimum number of samples required to split an internal node. If the value is too low, it can result in agressive splitting -> Overfit. If the value is too high -> Underfit.
3. Min Samples Leaf: It controls the minimum number of samples required to be at a leaf node... ensuring such split leaaves will at least have taht many samples in both branches. In common terms, it means that it rejects the splitting of a parent node which whould leave to its children having less than the specified min number. Preventing overfitting with a small value, but with a larger value can result in underfitting.

**Task 7**: Select a few different values for the three parameters you previously explored and train multiple Decision Tree models using these combinations. 

- Evaluate each model using a performance metric of your choice (e.g., Accuracy or F1-Score) on both the training and validation sets

- Compare the results across the different parameter settings

- Identify which combination of parameters provides the best overall performance


<span style="color:red">(4 marks)</span>


### First trying to optimise for Max Depth

In [ ]:
for max_depth in [3, 5, 10, None]:
    model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    print(f"Max Depth: {max_depth}")
    print("Training Accuracy:", accuracy_score(y_train, y_pred_train))
    print("Validation Accuracy:", accuracy_score(y_val, y_pred_val))
    print("f1 Score (Training):", classification_report(y_train, y_pred_train))
    print("f1 Score (Validation):", classification_report(y_val, y_pred_val))
    print("\n")

**Bonus**: Use GridSearch and RandomSearch to find the best combination of parameters. 

<span style="color:red">(0.5 marks)</span>

**Bonus**: Try to improve the results as much as possible. Some ideas of things you could try:

- Testing which are the best features to train the algorithm
- Cross-validation
- Majority voting
- Other classification algorithms
- A combination of classification algorithms

<span style="color:red">(Up to 2 marks)</span>